# Task 2 — Gains & Declines in Happiness (Slopegraphs, Altair)

Builds the two Task 2 slopegraphs for the **Drivers of Happiness** site: **gains** on the left, **declines** on the right.

**Method note:** single WHR years are noisy, so instead of comparing 2006 to 2025 directly we compare **3-year endpoint windows** (average of 2006-2008 vs. average of 2023-2025) and require at least 2 observations in each window.

**Every qualifying country is available**: countries whose score rose (or stayed exactly even, drawn grey) live in the gains chart, countries whose score fell live in the declines chart. Each exported page has a **checkbox dropdown** (ordered biggest mover first, with a filter box and a reset link) defaulting to the top 5. The y-axis **re-zooms to the current selection**, endpoint labels are re-dodged in JavaScript on every change, and labels hide automatically above 15 selected countries so the chart stays readable.

The charts use the site theme from Task 1, with the green/red poles of the map's color scale encoding direction of change (the pair passes colorblind-separation and contrast checks, and country names are written directly on every line, so color is never the only cue).

**Running in Google Colab:** upload this notebook, run all cells — the data-loading cell will prompt you to upload `whr_2006_2025_cleaned.csv` (it lives in `charts/data/` in the repo). Running locally from the repo root works too and skips the upload. The dropdown only exists in the exported HTML pages, not in the inline previews.

In [ ]:
%pip install -q "altair>=5.0,<6"

## Load the data

Reads from the repo if the notebook is run locally; otherwise prompts for an upload in Colab.

In [ ]:
import os
import pandas as pd

CSV_PATH = "charts/data/whr_2006_2025_cleaned.csv"  # path when run from the repo root

if os.path.exists(CSV_PATH):
    df_raw = pd.read_csv(CSV_PATH)
elif os.path.exists("whr_2006_2025_cleaned.csv"):
    df_raw = pd.read_csv("whr_2006_2025_cleaned.csv")
else:
    from google.colab import files  # only exists in Colab
    print("Upload whr_2006_2025_cleaned.csv")
    uploaded = files.upload()
    df_raw = pd.read_csv(next(iter(uploaded)))

print(df_raw.shape)
df_raw[["year", "country", "region", "life_evaluation"]].head()

## Split countries into gainers and decliners

For every country with enough data in both endpoint windows, compute the change in average life evaluation. Gainers (plus exact-even countries, which draw grey) go to the gains chart sorted biggest gain first, decliners to the declines chart sorted biggest decline first.

In [ ]:
import math

START_LABEL, END_LABEL = "2006–08", "2023–25"
DEFAULT_N = 5
EVEN_EPS = 0.005          # |change| below this rounds to 0.00 → "even", drawn grey
MAX_LABELS = 15           # above this many selected countries, labels are hidden
LABEL_PX = 14             # target label spacing in pixels (converted to score units)
PLOT_H = 360

def movers(df):
    """All countries with enough data in both windows, split by direction."""
    d = df.dropna(subset=["life_evaluation"])
    start = d[d.year.between(2006, 2008)].groupby("country").life_evaluation.agg(["mean", "count"])
    end = d[d.year.between(2023, 2025)].groupby("country").life_evaluation.agg(["mean", "count"])
    m = start.join(end, lsuffix="_start", rsuffix="_end", how="inner")
    m = m[(m.count_start >= 2) & (m.count_end >= 2)]
    m["change"] = m.mean_end - m.mean_start
    m["direction"] = "even"
    m.loc[m.change >= EVEN_EPS, "direction"] = "up"
    m.loc[m.change <= -EVEN_EPS, "direction"] = "down"
    m = m.join(d.groupby("country").region.first())
    gains = m[m.direction.isin(["up", "even"])].sort_values("change", ascending=False)
    declines = m[m.direction == "down"].sort_values("change")
    return gains, declines

gains, declines = movers(df_raw)
print(f"gains pool: {len(gains)} countries ({(gains.direction == 'even').sum()} even), "
      f"declines pool: {len(declines)}")
pd.concat([gains.head(), declines.head()])[["mean_start", "mean_end", "change"]].round(2)

## Label dodging and chart data

Each country becomes two rows (one per endpoint). Endpoint labels get a small greedy **dodge** so near-tied values don't overlap, spaced in score units derived from a pixel target so the spacing stays right as the y-axis re-zooms. The same dodge is re-implemented in the exported page's JavaScript.

In [ ]:
def rounded_domain(lo, hi):
    return [math.floor((lo - 0.15) * 2) / 2, math.ceil((hi + 0.15) * 2) / 2]


def dodge(values, lo, hi, gap):
    """Nudge label y-positions apart so near-tied endpoints stay readable."""
    order = sorted(range(len(values)), key=lambda i: -values[i])
    out = dict.fromkeys(order)
    prev = hi + gap
    for i in order:
        y = min(values[i], prev - gap)
        out[i] = y
        prev = y
    low = min(out.values())
    if low < lo:
        out = {i: y + (lo - low) for i, y in out.items()}
    return [out[i] for i in range(len(values))]


def initial_state(m, default_countries):
    """Initial y-domain and dodged label rows for the default selection."""
    sub = m.loc[default_countries]
    lo, hi = rounded_domain(
        min(sub.mean_start.min(), sub.mean_end.min()),
        max(sub.mean_start.max(), sub.mean_end.max()),
    )
    gap = LABEL_PX * (hi - lo) / PLOT_H
    starts = dodge(sub.mean_start.tolist(), lo, hi, gap)
    ends = dodge(sub.mean_end.tolist(), lo, hi, gap)
    rows = []
    for (country, r), sy, ey in zip(sub.iterrows(), starts, ends):
        rows.append({"period": START_LABEL, "label_y": sy, "text": f"{r.mean_start:.1f}"})
        rows.append({"period": END_LABEL, "label_y": ey, "text": f"{country}  {r.mean_end:.1f}"})
    return [lo, hi], rows


def slope_data(m):
    """Long-form rows (two per country), ordered biggest mover first."""
    rows = []
    for country, r in m.iterrows():
        common = dict(country=country, region=r.region, change=r.change,
                      direction=r.direction, start_score=r.mean_start, end_score=r.mean_end)
        rows.append(dict(common, period=START_LABEL, score=r.mean_start))
        rows.append(dict(common, period=END_LABEL, score=r.mean_end))
    return pd.DataFrame(rows)


## Build the slopegraphs

Design choices:
- **Color encodes direction**: green up, grey even, red down — the poles of the Task 1 map scale. Identity is carried by direct labels on every line, never by color alone.
- The **y-domain is parameterized** (`ylo`/`yhi` signals) so the exported page can re-zoom the axis to whatever is selected.
- **2px lines, white-ringed endpoint dots**, recessive grid, and the site's font/ink via the same `site_theme()` as Task 1.
- The `sel` param holds a `|`-joined list of visible countries; the exported page's dropdown drives it. Label layers read a **named dataset** (`labels`) that the page rewrites on every change.
- **Tooltips** carry country, region, both window scores, and the signed change.

In [ ]:
import altair as alt

SITE_FONT = '-apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, "Helvetica Neue", Arial, sans-serif'
INK, INK_SOFT, LINE = "#2f3542", "#6b7280", "#ece4d4"
GREEN, RED, GREY = "#1a9850", "#d73027", "#9ca3af"

def site_theme(chart):
    return (
        chart.configure(font=SITE_FONT, background="white")
        .configure_view(stroke=None)
        .configure_title(
            anchor="start", color=INK, fontSize=17, fontWeight=700,
            subtitleColor=INK_SOFT, subtitleFontSize=12, subtitlePadding=6, offset=14,
        )
        .configure_axis(
            labelColor=INK_SOFT, titleColor=INK, labelFontSize=11, titleFontSize=12,
            gridColor=LINE, domainColor=LINE, tickColor=LINE,
        )
    )

def slopegraph(data, title, subtitle, default_countries, init_domain, init_labels):
    sel = alt.param(name="sel", value="|".join(default_countries))
    ylo = alt.param(name="ylo", value=init_domain[0])
    yhi = alt.param(name="yhi", value=init_domain[1])
    keep = 'indexof(split(sel, "|"), datum.country) >= 0'

    x = alt.X(
        "period:N", title=None, sort=[START_LABEL, END_LABEL],
        scale=alt.Scale(domain=[START_LABEL, END_LABEL], padding=0.35),
        axis=alt.Axis(labelAngle=0, labelFontSize=12, labelColor=INK, domain=False, ticks=False),
    )
    y_scale = alt.Scale(domainMin={"expr": "ylo"}, domainMax={"expr": "yhi"})
    y = alt.Y("score:Q", title="Life evaluation (0–10)",
              scale=y_scale, axis=alt.Axis(tickCount=5))
    color = alt.Color(
        "direction:N",
        scale=alt.Scale(domain=["up", "even", "down"], range=[GREEN, GREY, RED]),
        legend=None,
    )
    tooltip = [
        alt.Tooltip("country:N", title="Country"),
        alt.Tooltip("region:N", title="Region"),
        alt.Tooltip("start_score:Q", title=START_LABEL, format=".2f"),
        alt.Tooltip("end_score:Q", title=END_LABEL, format=".2f"),
        alt.Tooltip("change:Q", title="Change", format="+.2f"),
    ]

    base = alt.Chart(data)
    lines = base.mark_line(strokeWidth=2).encode(
        x=x, y=y, color=color, detail="country:N", tooltip=tooltip
    ).transform_filter(keep).add_params(sel, ylo, yhi)
    points = base.mark_point(filled=True, size=70, stroke="white", strokeWidth=1.5).encode(
        x=x, y=y, color=color, detail="country:N", tooltip=tooltip
    ).transform_filter(keep)

    # Label layers read the JS-managed dataset "labels", rewritten
    # (filtered + dodged) on every selection change by the page script.
    labels = alt.Chart(alt.Data(values=init_labels, name="labels"))
    left_vals = labels.transform_filter(f'datum.period == "{START_LABEL}"').mark_text(
        align="right", dx=-10, fontSize=11, color=INK_SOFT
    ).encode(x=x, y=alt.Y("label_y:Q", scale=y_scale), text="text:N")
    right_names = labels.transform_filter(f'datum.period == "{END_LABEL}"').mark_text(
        align="left", dx=10, fontSize=11, fontWeight=600, color=INK
    ).encode(x=x, y=alt.Y("label_y:Q", scale=y_scale), text="text:N")

    return (lines + points + left_vals + right_names).properties(
        width=250, height=PLOT_H,
        padding={"left": 5, "right": 95, "top": 5, "bottom": 5},
        title=alt.Title(title, subtitle=subtitle),
    )

SUBTITLE = "Average life evaluation, 3-year endpoint windows"

gains_default = list(gains.index[:DEFAULT_N])
declines_default = list(declines.index[:DEFAULT_N])

gains_domain, gains_labels = initial_state(gains, gains_default)
declines_domain, declines_labels = initial_state(declines, declines_default)

gains_chart = site_theme(slopegraph(
    slope_data(gains), "Biggest gains in happiness", SUBTITLE,
    gains_default, gains_domain, gains_labels))
declines_chart = site_theme(slopegraph(
    slope_data(declines), "Biggest declines in happiness", SUBTITLE,
    declines_default, declines_domain, declines_labels))

gains_chart

In [ ]:
declines_chart

## Export standalone HTML with the country dropdown

Each chart is wrapped in a page template that adds the themed checkbox dropdown (all countries in that direction, filter box, reset link, at least one country always kept on). Every change updates the `sel` signal, re-zooms `ylo`/`yhi`, and rewrites the `labels` dataset with freshly dodged positions.

In [ ]:
import json

PAGE_TEMPLATE = """<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="utf-8">
<title>__TITLE__</title>
<script src="https://cdn.jsdelivr.net/npm/vega@5"></script>
<script src="https://cdn.jsdelivr.net/npm/vega-lite@5"></script>
<script src="https://cdn.jsdelivr.net/npm/vega-embed@6"></script>
<style>
  body {
    font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto,
      "Helvetica Neue", Arial, sans-serif;
    margin: 0;
    padding: 14px 16px;
    background: #ffffff;
  }
  #picker { display: flex; align-items: center; gap: 8px; margin-bottom: 12px; position: relative; }
  #picker .picker-label { color: #6b7280; font-size: 12px; }
  #dd-btn {
    border: 1px solid #ece4d4;
    background: #fffbef;
    color: #2f3542;
    font: inherit;
    font-size: 12px;
    font-weight: 600;
    padding: 5px 14px;
    border-radius: 999px;
    cursor: pointer;
  }
  #dd-btn:hover { border-color: #f6b93b; }
  #dd-panel {
    position: absolute;
    top: 32px;
    left: 70px;
    z-index: 20;
    width: 240px;
    background: #ffffff;
    border: 1px solid #ece4d4;
    border-radius: 10px;
    box-shadow: 0 6px 18px rgba(47, 53, 66, 0.12);
    padding: 8px;
  }
  #dd-filter {
    width: 100%;
    box-sizing: border-box;
    border: 1px solid #ece4d4;
    border-radius: 6px;
    padding: 5px 8px;
    font: inherit;
    font-size: 12px;
    margin-bottom: 6px;
  }
  #dd-list { max-height: 210px; overflow-y: auto; }
  #dd-list label {
    display: flex;
    align-items: center;
    gap: 7px;
    font-size: 12px;
    color: #2f3542;
    padding: 4px 4px;
    border-radius: 6px;
    cursor: pointer;
  }
  #dd-list label:hover { background: #fffbef; }
  #dd-list input { accent-color: #f6b93b; margin: 0; }
  #dd-list .chg { margin-left: auto; color: #6b7280; font-variant-numeric: tabular-nums; }
  #dd-list .even .chg { color: #9ca3af; }
  #dd-reset {
    border: none;
    background: none;
    color: #e58e26;
    font: inherit;
    font-size: 12px;
    cursor: pointer;
    padding: 6px 4px 2px;
  }
  #note { color: #9ca3af; font-size: 11px; margin-left: 4px; }
</style>
</head>
<body>
<div id="picker">
  <span class="picker-label">Countries:</span>
  <button id="dd-btn" type="button"></button>
  <span id="note"></span>
  <div id="dd-panel" hidden>
    <input id="dd-filter" type="text" placeholder="Type to filter...">
    <div id="dd-list"></div>
    <button id="dd-reset" type="button">Reset to top 5</button>
  </div>
</div>
<div id="vis"></div>
<script>
  const spec = __SPEC__;
  const COUNTRIES = __COUNTRIES__;   // ordered biggest mover first
  const DEFAULT_N = __DEFAULT_N__;
  const START_LABEL = "__START__", END_LABEL = "__END__";
  const MAX_LABELS = __MAX_LABELS__, LABEL_PX = __LABEL_PX__, PLOT_H = __PLOT_H__;

  const selected = new Set(COUNTRIES.slice(0, DEFAULT_N).map(c => c.country));

  function roundedDomain(lo, hi) {
    return [Math.floor((lo - 0.15) * 2) / 2, Math.ceil((hi + 0.15) * 2) / 2];
  }

  // Same greedy dodge as the notebook: nudge labels apart top-down.
  function dodge(vals, lo, hi, gap) {
    const order = vals.map((v, i) => i).sort((a, b) => vals[b] - vals[a]);
    let prev = hi + gap;
    const out = {};
    for (const i of order) { out[i] = Math.min(vals[i], prev - gap); prev = out[i]; }
    const low = Math.min(...Object.values(out), hi);
    const shift = low < lo ? lo - low : 0;
    return vals.map((v, i) => out[i] + shift);
  }

  function currentState() {
    const active = COUNTRIES.filter(c => selected.has(c.country));
    const ys = active.flatMap(c => [c.start, c.end]);
    const [lo, hi] = roundedDomain(Math.min(...ys), Math.max(...ys));
    const rows = [];
    if (active.length <= MAX_LABELS) {
      const gap = LABEL_PX * (hi - lo) / PLOT_H;
      const starts = dodge(active.map(c => c.start), lo, hi, gap);
      const ends = dodge(active.map(c => c.end), lo, hi, gap);
      active.forEach((c, i) => {
        rows.push({ period: START_LABEL, label_y: starts[i], text: c.start.toFixed(1) });
        rows.push({ period: END_LABEL, label_y: ends[i], text: c.country + "  " + c.end.toFixed(1) });
      });
    }
    return { lo, hi, rows, n: active.length };
  }

  vegaEmbed("#vis", spec, { actions: false, renderer: "svg" }).then(({ view }) => {
    const btn = document.getElementById("dd-btn");
    const panel = document.getElementById("dd-panel");
    const list = document.getElementById("dd-list");
    const filter = document.getElementById("dd-filter");
    const note = document.getElementById("note");

    function refresh() {
      const s = currentState();
      btn.textContent = s.n + " of " + COUNTRIES.length + " selected  \\u25be";
      note.textContent = s.n > MAX_LABELS ? "labels hidden above " + MAX_LABELS + " countries" : "";
      view.signal("sel", [...selected].join("|"));
      view.signal("ylo", s.lo);
      view.signal("yhi", s.hi);
      view.data("labels", s.rows);
      view.runAsync();
    }

    for (const c of COUNTRIES) {
      const label = document.createElement("label");
      if (c.direction === "even") label.className = "even";
      const box = document.createElement("input");
      box.type = "checkbox";
      box.checked = selected.has(c.country);
      box.dataset.country = c.country;
      box.addEventListener("change", () => {
        if (box.checked) {
          selected.add(c.country);
        } else {
          if (selected.size === 1) { box.checked = true; return; }  // keep at least one
          selected.delete(c.country);
        }
        refresh();
      });
      const name = document.createElement("span");
      name.textContent = c.country;
      const chg = document.createElement("span");
      chg.className = "chg";
      chg.textContent = (c.change >= 0 ? "+" : "") + c.change.toFixed(2);
      label.append(box, name, chg);
      list.appendChild(label);
    }

    btn.addEventListener("click", () => { panel.hidden = !panel.hidden; });
    document.addEventListener("click", (e) => {
      if (!panel.hidden && !panel.contains(e.target) && e.target !== btn) panel.hidden = true;
    });
    filter.addEventListener("input", () => {
      const q = filter.value.toLowerCase();
      for (const row of list.children) {
        row.style.display = row.textContent.toLowerCase().includes(q) ? "" : "none";
      }
    });
    document.getElementById("dd-reset").addEventListener("click", () => {
      selected.clear();
      COUNTRIES.slice(0, DEFAULT_N).forEach(c => selected.add(c.country));
      for (const box of list.querySelectorAll("input")) {
        box.checked = selected.has(box.dataset.country);
      }
      refresh();
    });

    refresh();
  });
</script>
</body>
</html>
"""


def export(chart, m, filename, title):
    countries = [
        {"country": c, "start": round(r.mean_start, 2), "end": round(r.mean_end, 2),
         "change": round(r.change, 2), "direction": r.direction}
        for c, r in m.iterrows()
    ]
    html = (
        PAGE_TEMPLATE
        .replace("__TITLE__", title)
        .replace("__SPEC__", chart.to_json(indent=None))
        .replace("__COUNTRIES__", json.dumps(countries))
        .replace("__DEFAULT_N__", str(DEFAULT_N))
        .replace("__START__", START_LABEL)
        .replace("__END__", END_LABEL)
        .replace("__MAX_LABELS__", str(MAX_LABELS))
        .replace("__LABEL_PX__", str(LABEL_PX))
        .replace("__PLOT_H__", str(PLOT_H))
    )
    with open(filename, "w") as f:
        f.write(html)

export(gains_chart, gains, "task2_slopegraph_gains.html", "Biggest gains in happiness")
export(declines_chart, declines, "task2_slopegraph_declines.html", "Biggest declines in happiness")
print("Saved task2_slopegraph_gains.html and task2_slopegraph_declines.html")

try:
    from google.colab import files
    files.download("task2_slopegraph_gains.html")
    files.download("task2_slopegraph_declines.html")
except ImportError:
    pass  # running locally — files are next to the notebook

## Embedding into the site

Move the two exported files into `charts/`, then embed them side by side in Task 2's `viz-row` in `index.html`:

```html
<div class="viz-row">
  <iframe src="charts/task2_slopegraph_gains.html" class="viz-frame"
          loading="lazy" style="min-height: 580px"
          title="Countries with the biggest gains in happiness"></iframe>
  <iframe src="charts/task2_slopegraph_declines.html" class="viz-frame"
          loading="lazy" style="min-height: 580px"
          title="Countries with the biggest declines in happiness"></iframe>
</div>
```

Still to come for Task 2: the per-country explanatory-factors table.